In [ ]:
import requests
import pandas as pd
import unicodedata
from fuzzywuzzy import process

## Verificiar IDs grupo


In [ ]:
ZOTERO_USER_ID = "USER_ID"
ZOTERO_API_KEY = "API_KEY"

url = f"https://api.zotero.org/users/{ZOTERO_USER_ID}/groups"
headers = {"Zotero-API-Key": ZOTERO_API_KEY}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    groups = response.json()
    for group in groups:
        print(f"Grupo: {group['data']['name']} | Group ID: {group['data']['id']}")
else:
    print("Erro:", response.status_code, response.text)


## Adicionar qualis nos extras do zotero com um banco de dados qualis atualizado (2017-2020)

In [ ]:
# Configurações
ZOTERO_API_KEY = 'API_KEY'
ZOTERO_GROUP_ID = 'GROUP_ID'  # ID numérico do grupo
COLLECTION_KEY = 'COLLECTION_KEY'  # Chave da coleção

# 1. Carrega base Qualis
qualis_df = pd.read_csv(r"caminho/Qualis-2017-2020.csv",
                        sep=';', header=None, dtype=str, 
                        names=['ISSN', 'Nome', 'Área', 'Qualis'])

# Usa o melhor estrato para cada periódico
estrato_ordem = {
    'A1': 1,
    'A2': 2,
    'A3': 3,
    'A4': 4,
    'B1': 5,
    'B2': 6,
    'B3': 7,
    'B4': 8,
    'C': 9
}

qualis_dict = {}
for _, row in qualis_df.iterrows():
    nome = row['Nome'].strip().lower()
    estrato = row['Qualis'].strip().upper()
    if nome in qualis_dict:
        atual = qualis_dict[nome]
        if estrato_ordem.get(estrato, 99) < estrato_ordem.get(atual, 99):
            qualis_dict[nome] = estrato
    else:
        qualis_dict[nome] = estrato

# 2. Função para obter itens da coleção do grupo
def get_items_from_group_collection(group_id, collection_key):
    items = []
    start = 0
    while True:
        url = f"https://api.zotero.org/groups/{group_id}/collections/{collection_key}/items"
        headers = {"Zotero-API-Key": ZOTERO_API_KEY}
        params = {"limit": 100, "start": start}
        response = requests.get(url, headers=headers, params=params)
        if response.status_code != 200:
            print(f"Erro ao buscar itens da coleção (status {response.status_code}): {response.text}")
            break  # ou continue, dependendo do que você preferir
        batch = response.json()
        if not batch:
            break
        items.extend(batch)
        start += 100
    return items

# 3. Atualiza campo 'extra' com o Qualis
def update_item_extra_group(group_id, item_key, nota):
    url = f"https://api.zotero.org/groups/{group_id}/items/{item_key}"
    headers = {
        "Zotero-API-Key": ZOTERO_API_KEY,
        "Content-Type": "application/json"
    }
    # Obter dados do item
    get_response = requests.get(url, headers=headers)
    if get_response.status_code != 200:
        print(f"Erro ao buscar item {item_key}: {get_response.status_code}")
        return get_response.status_code
    item_data = get_response.json()


    # Atualiza/adiciona "Qualis: ..." no campo extra sem duplicar
    extra_atual = item_data['data'].get('extra', '')
    linhas_extra = [l for l in extra_atual.split('\n') if not l.strip().lower().startswith("qualis:")]
    linhas_extra.append(f"Qualis: {nota}")
    item_data['data']['extra'] = '\n'.join(linhas_extra)


    # Envia atualização
    put_response = requests.put(url, headers=headers, json=item_data)
    if put_response.status_code != 200:
        print(f"Erro ao atualizar item {item_key}: {put_response.status_code} - {put_response.text}")
    return response.status_code


# 4. Função para obter o melhor título do CSV com base na similaridade
def obter_qualis_mais_proximo(titulo_zotero, qualis_dict):
    # Encontra o título mais próximo do banco de dados CSV usando fuzzywuzzy
    titulo_normalizado = titulo_zotero.strip().lower()
    
    # Obtém a melhor correspondência e a similaridade
    melhor_correspondencia, similaridade = process.extractOne(titulo_normalizado, qualis_dict.keys())
    
    # Se a similaridade for maior que o limite (por exemplo, 80%)
    if similaridade >= 80:  # Limite de similaridade, ajuste conforme necessário
        return qualis_dict[melhor_correspondencia]
    return None  # Se não encontrar uma correspondência boa o suficiente

# 5. Roda o processo
items = get_items_from_group_collection(ZOTERO_GROUP_ID, COLLECTION_KEY)

for item in items:
    data = item.get('data', {})
    key = data.get('key')
    titulo_revista = data.get('publicationTitle')

    if not titulo_revista:
        continue
    # Usando a função de fuzzy matching para obter o melhor 'Qualis'
    nota = obter_qualis_mais_proximo(titulo_revista, qualis_dict)
    

    if nota:
        status = update_item_extra_group(ZOTERO_GROUP_ID, key, nota)
        print(f"Atualizado '{titulo_revista}' com Qualis {nota} - status {status}")
    else:
        print(f"Sem classificação Qualis para: {titulo_revista}")